# SMPT Verification

In [3]:
import smtplib
import dns.resolver
import re

def validate_email(email):
    email_regex = re.compile(r"[^@]+@[^@]+\.[^@]+")
    if not email_regex.match(email):
        return False, "Invalid email format"

    domain = email.split('@')[1]

    try:
        records = dns.resolver.resolve(domain, 'MX')
        mx_record = str(records[0].exchange)
        server = smtplib.SMTP(timeout=10)
        server.set_debuglevel(0)

        # SMTP Conversation
        server.connect(mx_record)
        server.helo(server.local_hostname)  #
        server.mail(email)
        code, message = server.rcpt(email)
        server.quit()

        # 250 indicates that the email address is valid
        if code == 250:
            return True, "Valid email address"
        else:
            return False, "Invalid email address"

    except (dns.resolver.NoAnswer, dns.resolver.NXDOMAIN):
        return False, "Domain not found"
    except smtplib.SMTPServerDisconnected:
        return False, "SMTP server disconnected"
    except smtplib.SMTPConnectError:
        return False, "Failed to connect to the SMTP server"
    except smtplib.SMTPRecipientsRefused:
        return False, "SMTP recipients refused"
    except Exception as e:
        return False, f"An error occurred: {str(e)}"


In [4]:
email = 'exxdandy@gmail.com'
is_valid, message = validate_email(email)
print(f"{email} - {message}")


exxdandy@gmail.com - An error occurred: [Errno 101] Network is unreachable
